# Entrenamiento del modelo Nex (v2, clases fusionadas)

Este notebook:
1. Descarga el dataset de Roboflow (`nex-cocina-oficial` v1, formato YOLOv11).
2. **Fusiona las 38 clases en 21** (`Jarra1..Jarra5` -> `Jarra`, `Persona1..6` -> `Persona`, etc.) sin volver a etiquetar.
3. Entrena YOLO11 y lo compara con el modelo actual (`jarras_nex_v1.pt`).
4. Descarga el modelo nuevo como `jarras_nex_v2.pt`.

**Antes de empezar:** menu *Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> GPU T4*.

> Generado por `training/build_notebook.py` del repositorio VISIONCOMPUTADORA-ATLAS. No editar a mano.

In [ ]:
# Misma version de Ultralytics que la PC del proyecto (requirements.txt)
!pip install -q ultralytics==8.4.176 roboflow
import torch
print("GPU disponible:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Descargar el dataset

**Opcion A (recomendada): directo desde Roboflow.** Guarda tu clave de API en los *Secretos* de Colab
(icono de llave a la izquierda) con el nombre `ROBOFLOW_API_KEY` y activa el acceso para este notebook.
Asi la clave **no queda escrita en el notebook**.

**Opcion B:** cambia `USAR_ROBOFLOW = False` y sube el zip exportado de Roboflow.

In [ ]:
USAR_ROBOFLOW = True

if USAR_ROBOFLOW:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    dataset = rf.workspace("alans-workspace-2f18i").project("nex-cocina-oficial").version(1).download("yolov11")
    ORIGEN = dataset.location
else:
    import zipfile
    from google.colab import files
    subido = files.upload()
    ORIGEN = "/content/dataset_roboflow"
    zipfile.ZipFile(next(iter(subido))).extractall(ORIGEN)

print("Dataset en:", ORIGEN)

## 2. Fusionar las clases (38 -> 21)

Las dos celdas siguientes crean los archivos del repositorio (`config/class_mapping.json` y
`scripts/merge_dataset_classes.py`). Si quieres cambiar como se agrupan las clases, edita la seccion
`"fusion"` del JSON.

In [ ]:
%%writefile class_mapping.json
{
  "descripcion": "Fusion de las 38 clases del dataset de Roboflow (nex-cocina-oficial v1) en clases unicas por TIPO de objeto. Las clases numeradas (Jarra1..Jarra5, Taper2..Taper5, Persona1..Persona6) identificaban objetos o personas especificas: el detector no puede distinguir jarras iguales por su aspecto; el seguimiento de cada objeto lo hace ByteTrack. Los nombres nuevos coinciden con los que ya usa src/analysis/kitchen.py, asi el sistema funciona igual con el modelo nuevo.",
  "clases": [
    "Jarra",
    "Jarra Licuadora",
    "Taper",
    "Olla",
    "Tapa de Olla",
    "Vaso",
    "Botella",
    "Cerveza",
    "Caja de Cerveza",
    "Cuchillo",
    "Cernidor",
    "Balde",
    "Envase Ketchup",
    "Envase Llajua",
    "Envase Salsa",
    "Envase Platanos",
    "Productos Culinarios",
    "Persona",
    "Licuadora",
    "Microondas",
    "Basurero"
  ],
  "fusion": {
    "Jarra 1": "Jarra",
    "Jarra1": "Jarra",
    "Jarra2": "Jarra",
    "Jarra3": "Jarra",
    "Jarra4": "Jarra",
    "Jarra5": "Jarra",
    "JarraLicuadora": "Jarra Licuadora",
    "Taper": "Taper",
    "Taper2": "Taper",
    "Taper3": "Taper",
    "Taper4": "Taper",
    "Taper5": "Taper",
    "olla": "Olla",
    "olla1": "Olla",
    "Tapa de olla": "Tapa de Olla",
    "Tapa1": "Tapa de Olla",
    "Vaso1": "Vaso",
    "botella": "Botella",
    "Cerveza": "Cerveza",
    "CajaCerveza": "Caja de Cerveza",
    "Cuchillo1": "Cuchillo",
    "Cernidor": "Cernidor",
    "Balde": "Balde",
    "Envase Ketchup": "Envase Ketchup",
    "Envase Llajua": "Envase Llajua",
    "Envase salsa": "Envase Salsa",
    "Envase para platanos": "Envase Platanos",
    "Productos culinarios": "Productos Culinarios",
    "Persona1": "Persona",
    "Persona2": "Persona",
    "Persona3": "Persona",
    "Persona4": "Persona",
    "Persona5": "Persona",
    "Persona6": "Persona",
    "Licuadora": "Licuadora",
    "Licuadora2": "Licuadora",
    "Microondas": "Microondas",
    "Basurero": "Basurero"
  }
}

In [ ]:
%%writefile merge_dataset_classes.py
"""
Fusiona las clases de un dataset YOLO (exportado de Roboflow) segun un archivo de mapeo.

No hace falta volver a etiquetar: solo se reescribe el numero de clase de cada
etiqueta (cajas o poligonos) y se genera un data.yaml nuevo.

Uso:
    python scripts/merge_dataset_classes.py --src datasets/nex_roboflow --dst datasets/nex_v2
    python scripts/merge_dataset_classes.py --src /content/nex-cocina-oficial-1 --dst /content/nex_v2 \\
        --mapping config/class_mapping.json

Este archivo solo usa la libreria estandar de Python y PyYAML (incluido con
Ultralytics), para poder ejecutarse tal cual en Google Colab.
"""

import argparse
import json
import os
import shutil
import sys
from collections import Counter
from pathlib import Path

import yaml

SPLITS = ("train", "valid", "test")
DEFAULT_MAPPING = Path(__file__).resolve().parent.parent / "config" / "class_mapping.json"


def load_mapping(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    new_names = data["clases"]
    fusion = data["fusion"]
    unknown_targets = sorted(set(fusion.values()) - set(new_names))
    if unknown_targets:
        raise ValueError(f"El mapeo apunta a clases que no estan en 'clases': {unknown_targets}")
    return new_names, fusion


def load_source_names(src):
    data = yaml.safe_load((Path(src) / "data.yaml").read_text(encoding="utf-8"))
    names = data["names"]
    return list(names.values()) if isinstance(names, dict) else list(names)


def build_index(old_names, new_names, fusion, drop_unmapped=False):
    """Devuelve {id_viejo: id_nuevo} (o None para descartar la clase)."""
    missing = [n for n in old_names if n not in fusion]
    if missing and not drop_unmapped:
        raise ValueError(
            f"Clases del dataset sin mapeo: {missing}. Agregalas a config/class_mapping.json "
            "o usa --drop-unmapped para descartarlas."
        )
    new_index = {name: i for i, name in enumerate(new_names)}
    return {i: (new_index[fusion[n]] if n in fusion else None) for i, n in enumerate(old_names)}


def remap_label_text(text, index):
    """Reescribe el numero de clase de cada fila (cajas o poligonos). Descarta clases None."""
    out, dropped = [], 0
    for row in text.splitlines():
        parts = row.split()
        if not parts:
            continue
        new_id = index.get(int(parts[0]))
        if new_id is None:
            dropped += 1
            continue
        out.append(" ".join([str(new_id)] + parts[1:]))
    return ("\n".join(out) + "\n") if out else "", dropped


def _link_or_copy(src, dst):
    """Usa un enlace duro (no ocupa espacio extra) y si no se puede, copia."""
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def merge_dataset(src, dst, mapping_path=DEFAULT_MAPPING, drop_unmapped=False):
    src, dst = Path(src), Path(dst)
    if dst.exists() and any(dst.iterdir()):
        raise FileExistsError(f"La carpeta destino '{dst}' ya existe y no esta vacia")

    new_names, fusion = load_mapping(mapping_path)
    old_names = load_source_names(src)
    index = build_index(old_names, new_names, fusion, drop_unmapped)

    counts = {s: Counter() for s in SPLITS}
    images = Counter()
    dropped = 0
    for split in SPLITS:
        img_dir, lbl_dir = src / split / "images", src / split / "labels"
        if not img_dir.is_dir():
            continue
        (dst / split / "images").mkdir(parents=True, exist_ok=True)
        (dst / split / "labels").mkdir(parents=True, exist_ok=True)
        for img in sorted(img_dir.iterdir()):
            if not img.is_file():
                continue
            _link_or_copy(img, dst / split / "images" / img.name)
            images[split] += 1
            lbl = lbl_dir / f"{img.stem}.txt"
            text = lbl.read_text(encoding="utf-8") if lbl.exists() else ""
            new_text, n_drop = remap_label_text(text, index)
            dropped += n_drop
            (dst / split / "labels" / f"{img.stem}.txt").write_text(new_text, encoding="utf-8")
            for row in new_text.splitlines():
                counts[split][new_names[int(row.split()[0])]] += 1

    data_yaml = {
        "path": str(dst.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images" if images["test"] else None,
        "nc": len(new_names),
        "names": new_names,
    }
    data_yaml = {k: v for k, v in data_yaml.items() if v is not None}
    (dst / "data.yaml").write_text(yaml.safe_dump(data_yaml, allow_unicode=True, sort_keys=False), encoding="utf-8")

    summary = {
        "origen": str(src),
        "clases_antes": len(old_names),
        "clases_despues": len(new_names),
        "imagenes": dict(images),
        "etiquetas_descartadas": dropped,
        "cajas_por_clase": {name: {s: counts[s][name] for s in SPLITS} for name in new_names},
    }
    (dst / "resumen_fusion.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    return summary


def print_summary(summary):
    print(f"Clases: {summary['clases_antes']} -> {summary['clases_despues']}")
    print(f"Imagenes: {summary['imagenes']}")
    if summary["etiquetas_descartadas"]:
        print(f"Etiquetas descartadas (clases sin mapeo): {summary['etiquetas_descartadas']}")
    print(f"\n{'Clase':<22}{'train':>8}{'valid':>8}{'test':>8}")
    for name, c in summary["cajas_por_clase"].items():
        warn = "  <- pocos ejemplos" if c["valid"] < 5 else ""
        print(f"{name:<22}{c['train']:>8}{c['valid']:>8}{c['test']:>8}{warn}")


def main():
    parser = argparse.ArgumentParser(description="Fusionar clases de un dataset YOLO")
    parser.add_argument("--src", required=True, help="Carpeta del dataset exportado (con data.yaml)")
    parser.add_argument("--dst", required=True, help="Carpeta nueva donde crear el dataset fusionado")
    parser.add_argument("--mapping", default=str(DEFAULT_MAPPING), help="Archivo JSON de mapeo de clases")
    parser.add_argument("--drop-unmapped", action="store_true",
                        help="Descartar las clases que no esten en el mapeo (por defecto es un error)")
    args = parser.parse_args()
    try:
        summary = merge_dataset(args.src, args.dst, args.mapping, args.drop_unmapped)
    except (ValueError, FileExistsError, FileNotFoundError) as exc:
        sys.exit(f"Error: {exc}")
    print_summary(summary)
    print(f"\nDataset fusionado en: {Path(args.dst).resolve()}")


if __name__ == "__main__":
    main()

In [ ]:
!rm -rf /content/nex_v2
!python merge_dataset_classes.py --src "{ORIGEN}" --dst /content/nex_v2 --mapping class_mapping.json

## 3. Entrenar

- `PESOS_INICIALES = "yolo11n.pt"` entrena desde el modelo base de Ultralytics (igual que v1).
- Si subes `jarras_nex_v1.pt` a Colab puedes usarlo como punto de partida (`"/content/jarras_nex_v1.pt"`):
  reutiliza lo aprendido de la cocina y suele converger mas rapido.
- Con GPU T4 tarda aproximadamente 15-30 minutos.

In [ ]:
from ultralytics import YOLO

PESOS_INICIALES = "yolo11n.pt"
EPOCAS = 150
TAMANO_IMAGEN = 640

modelo = YOLO(PESOS_INICIALES)
modelo.train(
    data="/content/nex_v2/data.yaml",
    epochs=EPOCAS,
    imgsz=TAMANO_IMAGEN,
    batch=16,
    patience=40,
    seed=0,
    project="/content/runs",
    name="nex_v2",
    exist_ok=True,
)

## 4. Comparar con el modelo actual

La linea base es `jarras_nex_v1.pt` evaluado sobre el **mismo** conjunto de validacion, contando sus
38 clases como las 21 fusionadas (asi se comparan en igualdad de condiciones).

In [ ]:
LINEA_BASE = {"mAP50": 0.772, "mAP50-95": 0.633, "Jarra mAP50": 0.929, "Jarra mAP50-95": 0.744}

mejor = YOLO("/content/runs/nex_v2/weights/best.pt")
val = mejor.val(data="/content/nex_v2/data.yaml", split="val", plots=True)
ap50 = dict(zip(val.box.ap_class_index.tolist(), val.box.ap50.tolist()))
ap = dict(zip(val.box.ap_class_index.tolist(), val.box.ap.tolist()))
jarra = list(mejor.names.values()).index("Jarra")

nuevo = {
    "mAP50": val.box.map50,
    "mAP50-95": val.box.map,
    "Jarra mAP50": ap50.get(jarra, 0.0),
    "Jarra mAP50-95": ap.get(jarra, 0.0),
}
print(f"{'Metrica':<16}{'v1 (actual)':>12}{'v2 (nuevo)':>12}")
for k in LINEA_BASE:
    print(f"{k:<16}{LINEA_BASE[k]:>12.3f}{nuevo[k]:>12.3f}")

In [ ]:
# Evaluacion final sobre el conjunto de prueba (imagenes que el modelo nunca vio)
mejor.val(data="/content/nex_v2/data.yaml", split="test")

## 5. Descargar el modelo nuevo

In [ ]:
import shutil
from google.colab import files

shutil.copy("/content/runs/nex_v2/weights/best.pt", "/content/jarras_nex_v2.pt")
files.download("/content/jarras_nex_v2.pt")

## 6. Usarlo en la PC del proyecto

1. Copia `jarras_nex_v2.pt` a `C:\PROYECTOSIA\VISIONCOMPUTADORA-ATLAS\models\custom\`.
2. En `config\cameras.json` cambia `"modelo_default"` a `"models/custom/jarras_nex_v2.pt"`.
3. Reinicia `Iniciar_Nex.bat`. Para volver al modelo anterior, deshaz el cambio del paso 2.

Los nombres de las clases nuevas son los mismos que el sistema ya usa, asi que no hay que cambiar codigo.